# Project 7: Training Classification Models with Noisy Labels

**Name:** [Your Name]  
**UW Email:** [your_email]@uwaterloo.ca  
**Course:** CS484/684 — Winter 2026

## Abstract

In real-world scenarios, training data often contains incorrectly labeled examples due to human annotation errors, automated labeling pipelines, or inherent ambiguity in the data. Standard training with cross-entropy loss is known to overfit to such noisy labels, degrading generalization performance. This project investigates robust training methods that can learn effectively despite label corruption.

We use a frozen DINO (ViT-B/16) encoder to extract high-quality image representations and train linear classifiers on top, isolating the effect of noisy labels from representation quality. We evaluate on CIFAR-10 with synthetically injected symmetric and asymmetric label noise at varying rates (0%–80%), comparing standard cross-entropy against robust alternatives: symmetric cross-entropy (SCE), forward loss correction via an estimated noise transition matrix, and confidence-based sample reweighting.

Beyond synthetic experiments, we evaluate on a naturally noisy fine-grained classification task where inter-class similarity leads to genuine annotation errors. Our analysis includes accuracy-vs-noise curves, confusion matrices, and ablation studies on key hyperparameters, providing insight into when and why robust methods succeed or fail.

## Team Members and Contributions

- **[Name A]** ([email]@uwaterloo.ca): [describe contributions]
- **[Name B]** ([email]@uwaterloo.ca): [describe contributions]

## Code Libraries

- **PyTorch** (`torch`, `torchvision`): Deep learning framework used for model definition, training, and DINO feature extraction.
- **NumPy**: Numerical operations including noise injection and matrix computations.
- **Matplotlib / Seaborn**: Visualization of training curves, accuracy plots, and confusion matrices.
- **scikit-learn**: Evaluation metrics (accuracy, confusion matrix) and data splitting utilities.

All libraries above are available via the default Anaconda distribution or standard pip install.

---
## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix

import os
import sys
sys.path.insert(0, 'mylibs')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

---
## 2. DINO Feature Extraction

We use a frozen pretrained DINO ViT-B/16 model as our feature encoder. By extracting and caching features once, we decouple representation learning from the noisy-label problem and can run all subsequent experiments efficiently on CPU.

In [ ]:
# Load pretrained DINO model
dino = torch.hub.load('facebookresearch/dino:main', 'dino_vitb16')
dino.eval()
dino.to(device)

# DINO expects 224x224 images with ImageNet normalization
transform_dino = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Load CIFAR-10
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform_dino)
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform_dino)

train_loader = DataLoader(trainset, batch_size=128, shuffle=False, num_workers=2)
test_loader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

CLASS_NAMES = trainset.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes: {CLASS_NAMES}')
print(f'Train: {len(trainset)}, Test: {len(testset)}')

In [ ]:
def extract_features(model, loader):
    """Extract features from a frozen model for all images in loader."""
    all_features = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            features = model(images)  # (B, 768) for ViT-B/16
            all_features.append(features.cpu())
            all_labels.append(labels)
    return torch.cat(all_features), torch.cat(all_labels)

# Extract and cache features
os.makedirs('features', exist_ok=True)

if os.path.exists('features/train_features.pt'):
    print('Loading cached features...')
    train_features = torch.load('features/train_features.pt')
    train_labels = torch.load('features/train_labels.pt')
    test_features = torch.load('features/test_features.pt')
    test_labels = torch.load('features/test_labels.pt')
else:
    print('Extracting train features...')
    train_features, train_labels = extract_features(dino, train_loader)
    print('Extracting test features...')
    test_features, test_labels = extract_features(dino, test_loader)
    torch.save(train_features, 'features/train_features.pt')
    torch.save(train_labels, 'features/train_labels.pt')
    torch.save(test_features, 'features/test_features.pt')
    torch.save(test_labels, 'features/test_labels.pt')

FEAT_DIM = train_features.shape[1]
print(f'Feature dim: {FEAT_DIM}, Train: {train_features.shape[0]}, Test: {test_features.shape[0]}')

---
## 3. Label Noise Injection

We implement two noise models:
- **Symmetric noise**: each label is independently flipped to a uniformly random other class with probability $p$.
- **Asymmetric noise**: labels are flipped to a semantically similar class with probability $p$ (e.g., truck $\to$ automobile, deer $\to$ horse).

In [ ]:
def inject_symmetric_noise(labels, noise_rate, num_classes=10):
    """Flip each label to a random other class with probability noise_rate."""
    noisy_labels = labels.clone()
    n = len(labels)
    num_noisy = int(noise_rate * n)
    noisy_idx = np.random.choice(n, num_noisy, replace=False)
    for idx in noisy_idx:
        original = noisy_labels[idx].item()
        choices = list(range(num_classes))
        choices.remove(original)
        noisy_labels[idx] = np.random.choice(choices)
    actual_rate = (noisy_labels != labels).float().mean().item()
    print(f'  Symmetric noise: requested={noise_rate:.0%}, actual={actual_rate:.2%}')
    return noisy_labels


# CIFAR-10 asymmetric noise: visually similar class pairs
ASYM_MAP = {
    9: 1,  # truck -> automobile
    2: 0,  # bird -> airplane
    3: 5,  # cat -> dog
    5: 3,  # dog -> cat
    4: 7,  # deer -> horse
}

def inject_asymmetric_noise(labels, noise_rate, asym_map=ASYM_MAP):
    """Flip labels to a similar class with probability noise_rate (only for mapped classes)."""
    noisy_labels = labels.clone()
    for src, dst in asym_map.items():
        mask = (labels == src)
        n_src = mask.sum().item()
        num_flip = int(noise_rate * n_src)
        if num_flip > 0:
            flip_idx = np.random.choice(np.where(mask.numpy())[0], num_flip, replace=False)
            noisy_labels[flip_idx] = dst
    actual_rate = (noisy_labels != labels).float().mean().item()
    print(f'  Asymmetric noise: requested={noise_rate:.0%}, actual={actual_rate:.2%}')
    return noisy_labels

---
## 4. Training Utilities

In [ ]:
def make_linear_classifier(feat_dim, num_classes):
    return nn.Linear(feat_dim, num_classes)


def train_model(model, train_features, train_labels, test_features, test_labels,
                loss_fn, epochs=50, lr=0.01, batch_size=256):
    """Train a linear classifier and return training history."""
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    train_ds = TensorDataset(train_features, train_labels)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    
    history = {'train_loss': [], 'test_acc': []}
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for feats, labels in train_loader:
            feats, labels = feats.to(device), labels.to(device)
            logits = model(feats)
            loss = loss_fn(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * feats.size(0)
        
        epoch_loss /= len(train_ds)
        history['train_loss'].append(epoch_loss)
        
        # Evaluate on clean test set
        model.eval()
        with torch.no_grad():
            logits = model(test_features.to(device))
            preds = logits.argmax(dim=1).cpu()
            acc = accuracy_score(test_labels.numpy(), preds.numpy())
        history['test_acc'].append(acc)
        
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1}/{epochs} — loss: {epoch_loss:.4f}, test acc: {acc:.4f}')
    
    return history


def evaluate(model, features, labels):
    model.eval()
    with torch.no_grad():
        logits = model(features.to(device))
        preds = logits.argmax(dim=1).cpu()
    return accuracy_score(labels.numpy(), preds.numpy())

---
## 5. Baseline: Standard Cross-Entropy with Clean and Noisy Labels

We first establish an upper bound (clean labels) and measure how standard CE degrades as noise increases.

In [ ]:
NOISE_RATES = [0.0, 0.2, 0.4, 0.6, 0.8]
ce_results = {}

for rate in NOISE_RATES:
    print(f'\n=== Noise rate: {rate:.0%} ===')
    if rate == 0.0:
        noisy_labels = train_labels.clone()
    else:
        noisy_labels = inject_symmetric_noise(train_labels, rate)
    
    model = make_linear_classifier(FEAT_DIM, NUM_CLASSES)
    loss_fn = nn.CrossEntropyLoss()
    history = train_model(model, train_features, noisy_labels,
                          test_features, test_labels, loss_fn)
    
    ce_results[rate] = history
    print(f'  Final test accuracy: {history["test_acc"][-1]:.4f}')

In [ ]:
# Plot baseline results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for rate, hist in ce_results.items():
    axes[0].plot(hist['test_acc'], label=f'{rate:.0%} noise')
    axes[1].plot(hist['train_loss'], label=f'{rate:.0%} noise')

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('CE Baseline: Test Accuracy vs. Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Training Loss')
axes[1].set_title('CE Baseline: Training Loss vs. Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('images/baseline_ce_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Robust Method 1: Symmetric Cross-Entropy (SCE)

SCE (Wang et al., 2019) combines standard CE with a reverse cross-entropy term:

$$\mathcal{L}_{SCE} = \alpha \cdot CE(p, q) + \beta \cdot CE(q, p)$$

where $q$ is the model's predicted distribution and $p$ is the one-hot label. The reverse CE term is naturally robust to noisy labels because it is bounded.

In [ ]:
class SymmetricCrossEntropy(nn.Module):
    def __init__(self, alpha=1.0, beta=1.0, num_classes=10):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.num_classes = num_classes
    
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        probs = torch.clamp(probs, min=1e-7, max=1.0)
        
        # Standard CE: -sum(p * log(q))
        ce = nn.functional.cross_entropy(logits, targets)
        
        # Reverse CE: -sum(q * log(p))
        one_hot = nn.functional.one_hot(targets, self.num_classes).float()
        one_hot = torch.clamp(one_hot, min=1e-4, max=1.0)
        rce = -torch.sum(probs * torch.log(one_hot), dim=1).mean()
        
        return self.alpha * ce + self.beta * rce

In [ ]:
# TODO: Run SCE experiments across noise rates and compare with CE baseline

---
## 7. Robust Method 2: Forward Loss Correction

Patrini et al. (2017) propose correcting the loss using an estimated noise transition matrix $T$, where $T_{ij} = P(\tilde{y}=j | y=i)$. The corrected loss uses the "forward" adjustment:

$$\mathcal{L}_{forward}(x, \tilde{y}) = -\log(T^\top \cdot p(x))_{\tilde{y}}$$

In [ ]:
# TODO: Implement transition matrix estimation and forward loss correction

---
## 8. Robust Method 3: Confidence-Based Sample Reweighting

After a warm-up period of standard training, we identify potentially noisy samples as those with high loss (the model is less confident about them). We then down-weight or discard these samples in subsequent training.

In [ ]:
# TODO: Implement confidence-based sample reweighting with warm-up

---
## 9. Comparison: All Methods on Synthetic Noise

In [ ]:
# TODO: Summary comparison plots — accuracy vs. noise rate for all methods

---
## 10. Asymmetric Noise Experiments

In [ ]:
# TODO: Repeat key experiments with asymmetric noise

---
## 11. Real-World Noisy Data Experiment

To validate our findings beyond synthetic noise, we evaluate on a dataset where annotation errors are expected due to high inter-class visual similarity.

In [ ]:
# TODO: Load real-world noisy dataset and run experiments

---
## 12. Ablation Studies

In [ ]:
# TODO: Ablations on hyperparameters (SCE alpha/beta, warm-up epochs, etc.)

---
## Conclusions

[TODO: 2-3 paragraphs summarizing findings]